# 02 · Backtest on 2024-25 (true out-of-sample)

Loads the saved bundle and evaluates it honestly against the Elo benchmark. Metrics: accuracy, Brier, log-loss, AUC, high-confidence accuracy (Wilson CI).

In [1]:
# Make the nba_pred package importable from notebooks/
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

In [2]:
from nba_pred.pipelines.backtest import run_backtest

r = run_backtest("2024-25")
m, e = r["model"], r["elo_benchmark"]
print("model (XGB+Elo):", {k: round(v,4) for k,v in m.items() if isinstance(v,(int,float))})
print("elo benchmark  :", {k: round(v,4) for k,v in e.items() if isinstance(v,(int,float))})
print("high-confidence:", m["high_conf"])

model (XGB+Elo): {'n': 1225, 'accuracy': 0.6612, 'brier': 0.2186, 'log_loss': 0.6276, 'auc': 0.7151, 'base_rate_home_win': 0.5445}
elo benchmark  : {'n': 1225, 'accuracy': 0.6653, 'brier': 0.2183, 'log_loss': 0.6289, 'auc': 0.719, 'base_rate_home_win': 0.5445}
high-confidence: {'threshold': 0.65, 'n': 441, 'coverage': 0.36, 'accuracy': 0.764172335600907, 'wilson_95': [0.7223751192990864, 0.801406829151778]}


### Confusion matrix & reliability

In [3]:
from nba_pred.pipelines.dataset import build_dataset
from nba_pred.model.bundle import ModelBundle
from nba_pred.evaluation import metrics, plots

bundle = ModelBundle.load()
X, y, meta = build_dataset(["2024-25"], __import__("nba_pred.config", fromlist=["DEFAULT"]).DEFAULT)
prob = bundle.predict_proba(X)
pred = (prob >= 0.5).astype(int)
plots.confusion_matrix_fig(y, pred, "2024-25: where does the model fail?").show()
plots.calibration_fig(metrics.calibration_table(y, prob), "2024-25 reliability").show()